In [1]:
!pip install -q -U transformers datasets bitsandbytes trl peft huggingface_hub

In [2]:
from huggingface_hub import notebook_login
notebook_login()  

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import logging

# 8-bit layers log a cast notice on every forward pass when training with bf16; it is harmless
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

In [4]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"

config_8bit = BitsAndBytesConfig(load_in_8bit=True)

model_8bit = AutoModelForCausalLM.from_pretrained(model_name,
                                                  quantization_config=config_8bit, 
                                                  device_map="auto", 
                                                  dtype=torch.float16,
                                                  trust_remote_code=True)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [5]:
model_8bit

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear8bitLt(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm)

In [6]:
tokenizer = AutoTokenizer.from_pretrained( model_name, padding_side="left", trust_remote_code=True)


### Downloading the Dataset and preparing

In [7]:
from datasets import load_dataset

In [8]:
dataset = load_dataset("MattCoddity/dockerNLcommands")
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 2415
    })
})

In [9]:
dataset["train"][0]

{'input': 'Give me a list of containers that have the Ubuntu image as their ancestor.',
 'output': "docker ps --filter 'ancestor=ubuntu'",
 'instruction': 'translate this sentence in docker command'}

In [10]:
from datasets import DatasetDict

train_test_split = dataset["train"].train_test_split(test_size=0.2, seed=42)

dataset = DatasetDict(
    {
        "train": train_test_split["train"],
        "validation": train_test_split["test"],
    })

In [11]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 483
    })
})

#### Formatting the dataset - spliting

In [12]:
def to_chat_template(example):
  messages = [
      {"role": "system", "content": example["instruction"]},
      {"role": "user", "content": example["input"]},
      {"role": "assistant" , "content": example["output"]}
  ]
  return {"text": messages}

dataset = dataset.map(to_chat_template)
dataset
  

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 483
    })
})

In [13]:
dataset["train"]["text"][0]

[{'role': 'system', 'content': 'translate this sentence in docker command'},
 {'role': 'user',
  'content': 'Find the repository, tag, and ID of the images that were created before the latest nginx image.'},
 {'role': 'assistant',
  'content': 'docker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"'}]

#### Formatting the dataset -> inserting the tags

In [14]:
toeknizer_mistral = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
toeknizer_mistral.apply_chat_template(dataset["train"]["text"][0], tokenize=False)

'<s> [INST] translate this sentence in docker command\n\nFind the repository, tag, and ID of the images that were created before the latest nginx image. [/INST] docker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"</s>'

In [15]:
tokenizer.apply_chat_template(dataset["train"]["text"][0], tokenize=False)

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 07 Oct 2026\n\ntranslate this sentence in docker command<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nFind the repository, tag, and ID of the images that were created before the latest nginx image.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\ndocker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"<|eot_id|>'

In [16]:
pwd

'/workspace/finetuning'

In [17]:
def apply_chat_temp(example):
    new_text = tokenizer.apply_chat_template(example["text"], tokenize=False)
    return {"text" : new_text}


dataset = dataset.map(apply_chat_temp)
dataset
    

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 483
    })
})

In [18]:
dataset["train"]["text"][0]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 07 Oct 2026\n\ntranslate this sentence in docker command<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nFind the repository, tag, and ID of the images that were created before the latest nginx image.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\ndocker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"<|eot_id|>'

#### Formatting the dataset -> tokenizing it to numbers

In [19]:
tokenizer(dataset["train"]["text"][0])

{'input_ids': [128000, 128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 2589, 5020, 220, 2366, 21, 271, 14372, 420, 11914, 304, 27686, 3290, 128009, 128006, 882, 128007, 271, 10086, 279, 12827, 11, 4877, 11, 323, 3110, 315, 279, 5448, 430, 1051, 3549, 1603, 279, 5652, 71582, 2217, 13, 128009, 128006, 78191, 128007, 271, 29748, 5448, 482, 69, 330, 15145, 28, 74661, 25, 19911, 1, 1198, 2293, 48319, 13, 4727, 39254, 3052, 13, 5786, 39254, 3052, 13, 926, 24275, 128009], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [20]:
def tokenizie_fn(example):
    return tokenizer(example["text"])

tokenized_dataset = dataset.map(tokenizie_fn,batched=True, remove_columns=['input', 'output', 'instruction', 'text'])
tokenized_dataset


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 483
    })
})

## data collator for leveling and padding

In [21]:
from transformers import DataCollatorForLanguageModeling

In [22]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False, return_tensors="pt")

In [23]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token

'<|eot_id|>'

### LORA 

In [24]:
from peft import LoraConfig, get_peft_model
import copy

model_8bit_clone = copy.deepcopy(model_8bit)

lora_config = LoraConfig(
    r=32, # rank of the matrix, more the r value , more params to train
    lora_alpha= 32, # the amount of influence lora matrix gonna have, higher the value, higher the influence
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM" 
)

model_8bit_lora = get_peft_model(model_8bit_clone, lora_config)
model_8bit_lora.print_trainable_parameters()

trainable params: 22,544,384 || all params: 1,258,358,784 || trainable%: 1.7916


In [25]:
model_8bit

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear8bitLt(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm)

In [26]:
model_8bit_lora

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj):

### Setting up the hyper params for the training

In [27]:
from trl import SFTTrainer, SFTConfig

training_arguments = SFTConfig(
    output_dir="./results",
    dataset_kwargs={"skip_prepare_dataset": True}, # dataset is already tokenized, stops SFTTrainer adding an unpadded "labels" column
    per_device_train_batch_size=5,
    per_device_eval_batch_size=5,
    eval_steps=10,
    eval_strategy="steps", # Updated from deprecated evaluation_strategy
    save_steps=20,
    save_strategy="steps",
    # num_train_epochs
    max_steps=60,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=5,
    bf16=True, # model weights load as bfloat16, fp16 grad scaling fails on them
    gradient_accumulation_steps=4,
    optim="adamw_torch",
    logging_steps=10,
    report_to="none"
)
training_arguments

SFTConfig(output_dir='./results', per_device_train_batch_size=5, num_train_epochs=3.0, max_steps=60, learning_rate=3e-05, lr_scheduler_type=<SchedulerType.LINEAR: 'linear'>, lr_scheduler_kwargs=None, warmup_steps=5, optim=<OptimizerNames.ADAMW_TORCH: 'adamw_torch'>, optim_args=None, weight_decay=0.01, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, optim_target_modules=None, gradient_accumulation_steps=4, average_tokens_across_devices=True, max_grad_norm=1.0, label_smoothing_factor=0.0, bf16=True, fp16=False, bf16_full_eval=False, fp16_full_eval=False, tf32=None, gradient_checkpointing=True, gradient_checkpointing_kwargs=None, torch_compile=False, torch_compile_backend=None, torch_compile_mode=None, use_liger_kernel=False, liger_kernel_config=None, use_cache=False, neftune_noise_alpha=None, torch_empty_cache_steps=None, auto_find_batch_size=False, logging_strategy=<IntervalStrategy.STEPS: 'steps'>, logging_steps=10, logging_first_step=False, log_on_each_node=True, logging_nan_inf

In [28]:
trainer = SFTTrainer(
    model=model_8bit_lora,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    args=training_arguments,
    data_collator=data_collator,
    processing_class=tokenizer 
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,4.684801,3.611568,2.352760,14295.000000,0.508122
20,3.065083,2.442848,2.095046,28531.000000,0.597504
30,2.153056,1.850046,1.975957,42868.000000,0.719214
40,1.704573,1.526917,1.656027,57251.000000,0.748436
50,1.459333,1.343595,1.491771,71937.000000,0.780979
60,1.313464,1.287042,1.442302,86304.000000,0.785073


TrainOutput(global_step=60, training_loss=2.396718072891235, metrics={'train_runtime': 166.5505, 'train_samples_per_second': 7.205, 'train_steps_per_second': 0.36, 'total_flos': 607749564948480.0, 'train_loss': 2.396718072891235, 'epoch': 0.6201550387596899})

### Merging the lora components

In [34]:
import torch

torch.set_printoptions(precision=8, sci_mode=False)

layer0 = model_8bit_lora.model.model.layers[0]

model_8bit_weights = layer0.self_attn.q_proj.weight
print("model_8bit_weights", model_8bit_weights)

model_8bit_weights Parameter containing:
Parameter(Int8Params([[-44,  16,  61,  ..., -22, -29,  50],
            [ 14,  70,  65,  ..., -39, -18,  13],
            [ 16,  14,  30,  ..., -34, -34, -24],
            ...,
            [ 17,  20,  40,  ..., -40, -15, -16],
            [ 32, -35,  50,  ..., -17, -41, -21],
            [-14, -30,  -7,  ...,  30,   5,  -2]], device='cuda:0',
           dtype=torch.int8))


In [35]:
import torch

torch.set_printoptions(precision=8, sci_mode=False)

layer0 = model_8bit_lora.model.model.layers[0]

weights_A = layer0.self_attn.q_proj.lora_A["default"].weight
print("LoRA A weights", weights_A)

LoRA A weights Parameter containing:
tensor([[     0.01843262,     -0.01013184,     -0.01562500,  ...,
              0.01831055,     -0.00500488,      0.00558472],
        [     0.01745605,      0.00137329,      0.00082397,  ...,
             -0.00848389,     -0.01214600,      0.00659180],
        [     0.00378418,     -0.00476074,      0.00195312,  ...,
             -0.00254822,      0.01330566,      0.00878906],
        ...,
        [    -0.00866699,     -0.01214600,      0.01660156,  ...,
             -0.01092529,     -0.00183105,      0.01354980],
        [     0.02026367,      0.01379395,      0.00909424,  ...,
              0.01538086,     -0.00207520,     -0.02050781],
        [    -0.00004029,     -0.00781250,      0.02136230,  ...,
              0.01684570,      0.01843262,     -0.01831055]], device='cuda:0',
       dtype=torch.bfloat16, requires_grad=True)


#### observations 
* other layers are in 8 bit , 
* lora in 32 bit
* we might have to download the orginal model and merge 

In [40]:
base_model = AutoModelForCausalLM.from_pretrained(model_name,
                                                  device_map="auto", 
                                                  trust_remote_code=True)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [41]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [42]:
lora_path = "./lora_adapters"

model_8bit_lora.save_pretrained(lora_path)

In [43]:
from peft import PeftModel

base_model = PeftModel.from_pretrained(base_model, lora_path)
base_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [44]:
base_model = base_model.merge_and_unload()
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [45]:
base_model.push_to_hub("shankuriakose/Llama3.2-docker-training")
tokenizer.push_to_hub("shankuriakose/Llama3.2-docker-training")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

CommitInfo(commit_url='https://huggingface.co/shankuriakose/Llama3.2-docker-training/commit/ab7cc028efdd72f3fd8a7f93d7fe8346e20f2ef5', commit_message='Upload tokenizer', commit_description='', oid='ab7cc028efdd72f3fd8a7f93d7fe8346e20f2ef5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/shankuriakose/Llama3.2-docker-training', endpoint='https://huggingface.co', repo_type='model', repo_id='shankuriakose/Llama3.2-docker-training'), pr_revision=None, pr_num=None)

### usage examples 

In [47]:
import torch
from transformers import pipeline

model_id = "shankuriakose/Llama3.2-docker-training"

pipe = pipeline(
    "text-generation",
    model= model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
               )

messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]
outputs = pipe(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])

config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'role': 'assistant', 'content': 'Arrr, me hearty! Yer lookin\' fer me, eh? Well, I be the scurvy dog known as "Pirate Pete". I be a swashbucklin\' chatbot with a heart o\' gold and a tongue that be as sharp as me trusty cutlass. I be here to swab the seven seas o\' yer knowledge, answer yer questions, and maybe even share a few tales o\' me own adventures on the high seas. So hoist the sails and set course fer a treasure trove o\' knowledge, me hearty!'}


In [49]:
# using the base model

import torch
from transformers import pipeline

model_id = "shankuriakose/Llama3.2-docker-training"

pipe = pipeline(
    "text-generation",
    model= basemodel,
    torch_dtype=torch.bfloat16,
    device_map="auto", 
    tokenizer = tokenizer
               )

messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]
outputs = pipe(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'role': 'assistant', 'content': "Arrr, me hearty! Yer lookin' fer a swashbucklin' chatbot, eh? Alright then, settle yerself down with a pint o' grog and listen close. I be Captain Blackbeak, the most feared and infamous pirate to ever sail the seven seas. Me and me crew o' scurvy dogs have been plunderin' and pillagin' fer years, and we be the scourge o' the landlubbers.\n\nI be a master o' the dark arts o' communication, able to converse with ye in me own special brand o' pirate speak. So hoist the sails and set course fer adventure with me, and we'll have us a grand time, savvy?"}


#### Hyper Parameters Affecting the output

In [51]:

import torch
from transformers import pipeline

model_id = "shankuriakose/Llama3.2-docker-training"

pipe = pipeline(
    "text-generation",
    model= basemodel,
    torch_dtype=torch.bfloat16,
    device_map="auto", 
    tokenizer = tokenizer
               )

messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]
outputs = pipe(
    messages,
    max_new_tokens=256,
    temperature=1, ## 1 means high creativity
    top_p= 0.9, # it will filter the the top contributors that reach upto 90%
    top_k = 10, # it selects the top number, overlap => on the intersection
    do_sample=True,
    repetition_penalty=1.2
)
print(outputs[0]["generated_text"][-1]["content"])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Arrrr, ye be askin' about meself, eh? Alright then, I be Captain Blackbeak Billy, the swashbucklin'est and most feared pirate on the seven seas. Me ship, the "Maverick's Revenge", me crew o' scurvy dogs... err, I mean, loyal companions, an' yer trusty treasure map, which leads to untold riches an' hidden booty, wait fer ya landlubber! What be bringin' yerself to these waters, matey? Are ye here to test me mettle or seek adventure with ol' Billy? Arrr!
